# Chapter 7.4: FlashAttention — Algorithm & Tiling Strategy

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain why standard attention is memory-bound** and quantify the bottleneck
2. **Understand the FlashAttention algorithm**: tiling + online softmax
3. **Analyze memory access patterns** and IO complexity
4. **Compare FlashAttention 1, 2, and 3** improvements
5. **Implement FlashAttention forward pass** in Triton step by step
6. **Add causal masking** to the implementation
7. **Support GQA/MQA** in FlashAttention

## Prerequisites

- Chapter 7.2 (Triton Programming)
- Chapter 7.3 (PagedAttention — especially online softmax)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.4_flash_attention.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.4_flash_attention.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Memory-Bound Attention Problem

### 1.1 Standard Attention Computation

Standard attention computes:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The naive implementation materializes the full attention matrix $S = QK^T / \sqrt{d_k}$ of size $(N \times N)$:

```python
# Step 1: Compute S = Q @ K^T — writes N×N matrix to HBM
S = Q @ K.transpose(-2, -1) / sqrt(d_k)    # O(N²d) compute, O(N²) memory

# Step 2: Read S from HBM, compute softmax, write P back to HBM  
P = softmax(S, dim=-1)                      # O(N²) compute, O(N²) memory

# Step 3: Read P from HBM, compute P @ V
O = P @ V                                    # O(N²d) compute, O(Nd) memory
```

### 1.2 The Problem: O(N²) Memory

For a sequence of length N = 4096 with d = 128:
- Attention matrix S: 4096 × 4096 × 4 bytes = **64 MB** per head
- With 32 heads: **2 GB** just for the attention matrix!
- This is read/written **3 times** (compute S, softmax, compute O)
- Total memory traffic: ~6 GB per attention layer

In [ ]:
# Quantify the memory bottleneck
def analyze_attention_memory(seq_len, num_heads, head_dim, dtype_bytes=2):
    """
    Analyze memory requirements and IO for standard vs FlashAttention.
    """
    print(f"Configuration: seq_len={seq_len}, heads={num_heads}, head_dim={head_dim}, FP{dtype_bytes*8}")
    print("=" * 70)
    
    # Standard attention
    qkv_memory = 3 * seq_len * head_dim * dtype_bytes * num_heads
    attn_matrix = seq_len * seq_len * dtype_bytes * num_heads
    output_memory = seq_len * head_dim * dtype_bytes * num_heads
    
    # IO analysis (reads + writes to HBM)
    standard_io = (
        2 * qkv_memory +  # Read Q, K for S = QK^T; Read V for O = PV
        3 * attn_matrix +  # Write S, Read S for softmax, Write P, Read P for O=PV
        output_memory  # Write O
    )
    
    # FlashAttention IO
    # Only reads Q, K, V once and writes O once
    # Plus O(N²d/M) factor for re-reading Q/K/V in tiles
    M = 192 * 1024  # ~192 KB of SRAM (shared memory)
    flash_io = qkv_memory + output_memory + (seq_len * seq_len * head_dim * dtype_bytes * num_heads) / (M // (dtype_bytes * head_dim))
    # Simplified: flash_io ≈ O(N²d²/M) which is much less than O(N²)
    flash_io_simple = 2 * qkv_memory + output_memory  # Best case: each read once
    
    print(f"\nMemory footprint:")
    print(f"  Q, K, V:            {qkv_memory/1e6:>10.1f} MB")
    print(f"  Attention matrix:   {attn_matrix/1e6:>10.1f} MB")
    print(f"  Output:             {output_memory/1e6:>10.1f} MB")
    print(f"  Standard total:     {(qkv_memory + attn_matrix + output_memory)/1e6:>10.1f} MB")
    print(f"  FlashAttention:     {(qkv_memory + output_memory)/1e6:>10.1f} MB (no attn matrix!)")
    print(f"  Memory savings:     {attn_matrix / (qkv_memory + attn_matrix + output_memory) * 100:.0f}%")
    
    print(f"\nHBM IO (total reads + writes):")
    print(f"  Standard:           {standard_io/1e9:>10.3f} GB")
    print(f"  FlashAttention:     {flash_io_simple/1e9:>10.3f} GB (best case)")
    print(f"  IO reduction:       {standard_io/flash_io_simple:>10.1f}x")
    
    # Time estimate on A100
    a100_bw = 2e12  # 2 TB/s
    print(f"\nEstimated time (A100, memory-bound):")
    print(f"  Standard:           {standard_io/a100_bw*1000:>10.3f} ms")
    print(f"  FlashAttention:     {flash_io_simple/a100_bw*1000:>10.3f} ms")

analyze_attention_memory(seq_len=2048, num_heads=32, head_dim=128)
print()
analyze_attention_memory(seq_len=8192, num_heads=32, head_dim=128)

## 2. FlashAttention Algorithm

### 2.1 Core Idea: Tiling + Online Softmax

FlashAttention avoids materializing the N×N attention matrix by:

1. **Tiling**: Process Q, K, V in blocks that fit in SRAM (shared memory)
2. **Online softmax**: Compute softmax incrementally as we process K/V blocks
3. **Never write the attention matrix**: Keep intermediate results in SRAM

### 2.2 Algorithm (Forward Pass)

```
Input: Q, K, V ∈ R^{N×d}
Output: O ∈ R^{N×d}

Split Q into Tr blocks of size Br: Q1, Q2, ..., Q_Tr
Split K, V into Tc blocks of size Bc: K1/V1, K2/V2, ..., K_Tc/V_Tc

For i = 1 to Tr:           # Outer loop: over Q blocks
    Load Qi from HBM to SRAM
    Initialize: Oi = 0, mi = -inf, li = 0
    
    For j = 1 to Tc:       # Inner loop: over K/V blocks
        Load Kj, Vj from HBM to SRAM
        
        Compute Sij = Qi @ Kj^T / sqrt(d)    [in SRAM!]
        
        # Online softmax update
        mij = max(Sij)                        # Block-wise max
        mi_new = max(mi, mij)                 # Running max
        Pij = exp(Sij - mi_new)               # Softmax numerator
        li = li * exp(mi - mi_new) + sum(Pij) # Running sum
        
        # Update output
        Oi = Oi * exp(mi - mi_new) + Pij @ Vj  # Rescale + accumulate
        mi = mi_new
    
    Oi = Oi / li  # Final normalization
    Write Oi to HBM
```

### 2.3 Why This Works

The key insight is that softmax can be decomposed:
$$\text{softmax}([x_1, x_2]) = \left[\frac{e^{x_1-m}}{e^{x_1-m} + e^{x_2-m}}, \frac{e^{x_2-m}}{e^{x_1-m} + e^{x_2-m}}\right]$$

When we see $x_2$ after $x_1$, we can update the running statistics:
- If $x_2 > x_1$: rescale previous results by $e^{x_1 - x_2}$
- If $x_2 \leq x_1$: new contribution is scaled by $e^{x_2 - x_1}$

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, axes = plt.subplots(1, 3, figsize=(16, 6))fig.suptitle('FlashAttention Tiling Strategy', fontsize=14, fontweight='bold', color='#2C3E50')# Q matrixax = axes[0]ax.set_xlim(0, 5)ax.set_ylim(0, 8)ax.set_title('Q Matrix (N x d)', fontsize=11, fontweight='bold', color=BLUE)ax.axis('off')N_blocks = 4for i in range(N_blocks):    color = BLUE if i == 1 else LIGHT_BLUE    alpha = 0.9 if i == 1 else 0.3    rect = plt.Rectangle((0.5, 6.5 - i*1.5), 4, 1.2, facecolor=color, alpha=alpha,                          edgecolor='white', lw=2)    ax.add_patch(rect)    label = f'Q block {i} (Br x d)' + (' <- current' if i == 1 else '')    ax.text(2.5, 7.1 - i*1.5, label, ha='center', fontsize=8, color='white' if i==1 else DARK_GRAY,            fontweight='bold')ax.annotate('Outer loop\n(over Q blocks)', xy=(4.8, 5), fontsize=9, color=BLUE,            fontweight='bold', ha='center')# K,V matricesax = axes[1]ax.set_xlim(0, 5)ax.set_ylim(0, 8)ax.set_title('K, V Matrices (N x d)', fontsize=11, fontweight='bold', color=GREEN)ax.axis('off')for i in range(N_blocks):    color = GREEN if i == 2 else LIGHT_GREEN    alpha = 0.9 if i == 2 else 0.3    rect = plt.Rectangle((0.5, 6.5 - i*1.5), 4, 1.2, facecolor=color, alpha=alpha,                          edgecolor='white', lw=2)    ax.add_patch(rect)    label = f'K/V block {i} (Bc x d)' + (' <- current' if i == 2 else '')    ax.text(2.5, 7.1 - i*1.5, label, ha='center', fontsize=8,            color='white' if i==2 else DARK_GRAY, fontweight='bold')ax.annotate('Inner loop\n(over K/V blocks)', xy=(4.8, 5), fontsize=9, color=GREEN,            fontweight='bold', ha='center')# SRAM computationax = axes[2]ax.set_xlim(0, 5)ax.set_ylim(0, 8)ax.set_title('In SRAM (On-Chip)', fontsize=11, fontweight='bold', color=RED)ax.axis('off')sram_items = [    (2.5, 7, 'Q tile (Br x d)', BLUE),    (2.5, 5.8, 'K tile (Bc x d)', GREEN),    (2.5, 4.6, 'S = Q @ K^T (Br x Bc)', ORANGE),    (2.5, 3.4, 'P = softmax(S)', ORANGE),    (2.5, 2.2, 'V tile (Bc x d)', GREEN),    (2.5, 1, 'O += P @ V (Br x d)', RED),]for x, y, label, color in sram_items:    box = mpatches.FancyBboxPatch((0.3, y-0.4), 4.4, 0.8, boxstyle="round,pad=0.05",                                   facecolor=color, alpha=0.7, edgecolor='white', lw=1.5)    ax.add_patch(box)    ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')plt.tight_layout(rect=[0, 0.05, 1, 0.93])fig.text(0.5, 0.01, 'Attention matrix S (N x N) is NEVER fully materialized -- only Br x Bc tiles exist in SRAM',         ha='center', fontsize=10, color=RED, fontweight='bold')plt.savefig("flash_attention_tiling.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
import torch
import numpy as np

def flash_attention_reference(Q, K, V, Br=32, Bc=32):
    """
    Reference FlashAttention implementation in PyTorch.
    
    This follows the algorithm exactly, processing Q and K/V in tiles.
    The attention matrix is NEVER fully materialized.
    
    Args:
        Q, K, V: (N, d) tensors
        Br: block size for Q (rows)
        Bc: block size for K/V (columns)
    """
    N, d = Q.shape
    scale = 1.0 / (d ** 0.5)
    
    O = torch.zeros_like(Q)
    
    # Track max SRAM usage
    max_sram_elements = 0
    
    # Outer loop: over Q blocks
    for i in range(0, N, Br):
        i_end = min(i + Br, N)
        Qi = Q[i:i_end]  # (Br, d)
        
        # Online softmax state for this Q block
        mi = torch.full((i_end - i, 1), float('-inf'), device=Q.device)
        li = torch.zeros((i_end - i, 1), device=Q.device)
        Oi = torch.zeros((i_end - i, d), device=Q.device)
        
        # Inner loop: over K/V blocks
        for j in range(0, N, Bc):
            j_end = min(j + Bc, N)
            Kj = K[j:j_end]  # (Bc, d)
            Vj = V[j:j_end]  # (Bc, d)
            
            # SRAM usage: Qi + Kj + Vj + Sij + Pij
            sram = (i_end-i)*d + 2*(j_end-j)*d + (i_end-i)*(j_end-j)*2
            max_sram_elements = max(max_sram_elements, sram)
            
            # Compute attention scores (in SRAM!)
            Sij = (Qi @ Kj.T) * scale  # (Br, Bc)
            
            # Online softmax update
            mij = Sij.max(dim=-1, keepdim=True).values  # (Br, 1)
            mi_new = torch.maximum(mi, mij)
            
            # Rescale previous state
            alpha = torch.exp(mi - mi_new)  # Rescale factor
            Pij = torch.exp(Sij - mi_new)   # New softmax numerators
            
            # Update running sum
            li = li * alpha + Pij.sum(dim=-1, keepdim=True)
            
            # Update output
            Oi = Oi * alpha + Pij @ Vj
            
            mi = mi_new
        
        # Final normalization
        O[i:i_end] = Oi / li
    
    print(f"Max SRAM usage: {max_sram_elements * 4 / 1024:.1f} KB (float32)")
    return O


if torch.cuda.is_available():
    N, d = 256, 64
    Q = torch.randn(N, d, device='cuda')
    K = torch.randn(N, d, device='cuda')
    V = torch.randn(N, d, device='cuda')
    
    # FlashAttention reference
    O_flash = flash_attention_reference(Q, K, V, Br=64, Bc=64)
    
    # Standard attention
    scale = 1.0 / (d ** 0.5)
    S = Q @ K.T * scale
    P = torch.softmax(S, dim=-1)
    O_standard = P @ V
    
    max_error = (O_flash - O_standard).abs().max().item()
    print(f"\nMax error (Flash vs Standard): {max_error:.2e}")
    print(f"PASSED: {max_error < 1e-4}")
else:
    print("GPU not available")

## 3. IO Complexity Analysis

### 3.1 Standard Attention IO

$$\text{IO}_{\text{standard}} = \Theta(Nd + N^2)$$

The $N^2$ term dominates for typical LLM parameters ($N \gg d$).

### 3.2 FlashAttention IO

$$\text{IO}_{\text{flash}} = \Theta\left(\frac{N^2 d^2}{M}\right)$$

Where $M$ is the SRAM size. Since $M \gg d^2$ on modern GPUs:

$$\text{IO}_{\text{flash}} \ll \text{IO}_{\text{standard}}$$

### 3.3 Concrete Example

| Parameter | Value |
|-----------|-------|
| N (seq len) | 4096 |
| d (head dim) | 128 |
| M (SRAM) | 192 KB |

Standard: $4096 \times 128 + 4096^2 = 17.3M$ elements → **69 MB**

Flash: $\frac{4096^2 \times 128^2}{192 \times 1024 / 4} = 5.6M$ elements → **22 MB**

**3x less IO!**

In [ ]:
import matplotlib.pyplot as pltimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(10, 6))seq_lens = [512, 1024, 2048, 4096, 8192]d = 128M = 192 * 1024 // 4  # SRAM in float32 elementsstandard_io = [N*d + N*N for N in seq_lens]flash_io = [N*d + N*N*d*d/M for N in seq_lens]x = np.arange(len(seq_lens))width = 0.35bars1 = ax.bar(x - width/2, [io/1e6 for io in standard_io], width,               label='Standard Attention', color=RED, alpha=0.85, edgecolor='white', lw=1.5)bars2 = ax.bar(x + width/2, [io/1e6 for io in flash_io], width,               label='FlashAttention', color=GREEN, alpha=0.85, edgecolor='white', lw=1.5)ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')ax.set_ylabel('HBM IO (Million Elements)', fontsize=12, fontweight='bold')ax.set_title('HBM IO: Standard Attention vs FlashAttention', fontsize=14, fontweight='bold', color='#2C3E50')ax.set_xticks(x)ax.set_xticklabels(seq_lens)ax.legend(fontsize=11)ax.grid(axis='y', alpha=0.3)# Add reduction labelsfor i in range(len(seq_lens)):    ratio = standard_io[i] / flash_io[i]    ax.text(i, max(standard_io[i]/1e6, flash_io[i]/1e6) * 1.05,            f'{ratio:.1f}x', ha='center', fontsize=10, fontweight='bold', color=DARK_GRAY)ax.set_yscale('log')plt.tight_layout()plt.savefig("flash_attention_io.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Visualize IO complexity
import numpy as np

def plot_io_comparison():
    """Compare IO complexity of standard vs FlashAttention."""
    d = 128
    M = 192 * 1024 // 4  # SRAM in float32 elements
    
    seq_lens = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
    
    print(f"{'Seq Len':>8} | {'Standard IO':>14} | {'Flash IO':>14} | {'Reduction':>10}")
    print("-" * 55)
    
    for N in seq_lens:
        standard_io = N * d + N * N  # O(Nd + N²)
        flash_io = N * d + (N * N * d * d) / M  # O(Nd + N²d²/M)
        
        # In bytes (float16)
        standard_bytes = standard_io * 2
        flash_bytes = flash_io * 2
        
        ratio = standard_bytes / flash_bytes
        print(f"{N:>8} | {standard_bytes/1e6:>11.1f} MB | {flash_bytes/1e6:>11.1f} MB | {ratio:>9.2f}x")

plot_io_comparison()

## 4. FlashAttention-2 Improvements

### 4.1 Key Changes from v1 to v2

| Aspect | FlashAttention-1 | FlashAttention-2 |
|--------|-----------------|------------------|
| **Loop order** | Outer: K/V, Inner: Q | Outer: Q, Inner: K/V |
| **Parallelism** | Over batch × heads | Over batch × heads × seq_len |
| **Work per thread block** | One K/V block, all Q blocks | One Q block, all K/V blocks |
| **Shared memory writes** | Atomic updates to O | No atomic needed |
| **Warp specialization** | All warps do same work | Split warps for Q vs K/V |

### 4.2 Why the Loop Order Matters

**v1**: Each thread block processes one K/V block against ALL Q blocks
- Must write partial results for each Q block → atomics needed
- Limited parallelism: num_programs = num_KV_blocks × batch × heads

**v2**: Each thread block processes one Q block against ALL K/V blocks
- Each Q block has its own output → no atomics
- More parallelism: num_programs = num_Q_blocks × batch × heads
- Better GPU utilization for long sequences

### 4.3 Result

FlashAttention-2 achieves **2x speedup** over v1 and reaches **~70% of theoretical peak FLOPS** on A100.

## 5. FlashAttention-3 on Hopper (H100)

### 5.1 Hopper-Specific Optimizations

FlashAttention-3 exploits new H100 hardware features:

| Feature | How FA3 Uses It |
|---------|----------------|
| **TMA (Tensor Memory Accelerator)** | Async bulk memory copies, freeing compute warps |
| **WGMMA (Warp Group MMA)** | Larger matrix multiply units (64×64) |
| **FP8** | E4M3/E5M2 with FP32 accumulation |
| **Warp specialization** | Producer warps (data loading) + consumer warps (compute) |
| **Persistent kernels** | Kernel stays resident, processing multiple tiles |

### 5.2 Performance Gains

- **1.5-2.0x speedup** over FA2 on H100
- Reaches **~75% of peak FP16 FLOPS** (compared to ~50% for FA2 on H100)
- With FP8: up to **~2.5x** faster than FA2 FP16

## 6. Triton Implementation of FlashAttention

Now let's implement FlashAttention forward pass in Triton. We'll follow the FlashAttention-2 algorithm (outer loop over Q, inner loop over K/V).

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def flash_attention_fwd_kernel(
    # Inputs
    Q_ptr, K_ptr, V_ptr,
    # Output
    O_ptr,
    # Softmax statistics (for backward pass)
    L_ptr,  # log-sum-exp per row
    # Dimensions
    N,  # sequence length
    d,  # head dimension
    scale,  # 1/sqrt(d)
    # Strides
    stride_qn, stride_qd,
    stride_kn, stride_kd,
    stride_vn, stride_vd,
    stride_on, stride_od,
    # Causal masking
    IS_CAUSAL: tl.constexpr,
    # Block sizes
    BLOCK_M: tl.constexpr,  # Block size for Q (rows)
    BLOCK_N: tl.constexpr,  # Block size for K/V (columns)
    BLOCK_D: tl.constexpr,  # Block size for head dimension
):
    """
    FlashAttention forward pass (single head, single batch element).
    
    Each program processes one block of Q rows (BLOCK_M rows).
    It iterates over ALL K/V blocks to compute the full attention for those Q rows.
    """
    # Which Q block does this program handle?
    pid = tl.program_id(0)
    
    # ---- Initialize pointers for this Q block ----
    q_block_start = pid * BLOCK_M
    
    # Offsets within the block
    m_offsets = q_block_start + tl.arange(0, BLOCK_M)  # Q row indices
    d_offsets = tl.arange(0, BLOCK_D)                   # Head dim indices
    
    # Masks
    m_mask = m_offsets < N
    d_mask = d_offsets < d
    
    # ---- Load Q block: (BLOCK_M, BLOCK_D) ----
    q_ptrs = Q_ptr + m_offsets[:, None] * stride_qn + d_offsets[None, :] * stride_qd
    q = tl.load(q_ptrs, mask=m_mask[:, None] & d_mask[None, :], other=0.0)
    
    # ---- Online softmax state ----
    # m_i: running max per Q row — shape (BLOCK_M,)
    # l_i: running sum of exp per Q row — shape (BLOCK_M,)
    # acc: accumulated output — shape (BLOCK_M, BLOCK_D)
    m_i = tl.full((BLOCK_M,), float('-inf'), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, BLOCK_D), dtype=tl.float32)
    
    # ---- Inner loop: iterate over K/V blocks ----
    # For causal masking, we only need to go up to the current Q block
    if IS_CAUSAL:
        kv_end = min((pid + 1) * BLOCK_M, N)
    else:
        kv_end = N
    
    for j in range(0, kv_end, BLOCK_N):
        n_offsets = j + tl.arange(0, BLOCK_N)  # K/V row indices
        n_mask = n_offsets < N
        
        # Load K block: (BLOCK_N, BLOCK_D)
        k_ptrs = K_ptr + n_offsets[:, None] * stride_kn + d_offsets[None, :] * stride_kd
        k = tl.load(k_ptrs, mask=n_mask[:, None] & d_mask[None, :], other=0.0)
        
        # Compute S = Q @ K^T: (BLOCK_M, BLOCK_N)
        s = tl.dot(q, tl.trans(k)) * scale
        
        # Apply causal mask: S[i, j] = -inf if key_pos > query_pos
        if IS_CAUSAL:
            causal_mask = m_offsets[:, None] >= n_offsets[None, :]
            s = tl.where(causal_mask, s, float('-inf'))
        
        # Mask out-of-bounds positions
        s = tl.where(n_mask[None, :], s, float('-inf'))
        
        # ---- Online softmax update ----
        # Step 1: Find max of current block
        m_ij = tl.max(s, axis=1)  # (BLOCK_M,)
        
        # Step 2: New running max
        m_new = tl.maximum(m_i, m_ij)  # (BLOCK_M,)
        
        # Step 3: Rescale previous state
        alpha = tl.exp(m_i - m_new)  # (BLOCK_M,)
        
        # Step 4: Compute softmax numerators
        p = tl.exp(s - m_new[:, None])  # (BLOCK_M, BLOCK_N)
        
        # Step 5: Update running sum
        l_i = l_i * alpha + tl.sum(p, axis=1)  # (BLOCK_M,)
        
        # Step 6: Load V block and accumulate
        v_ptrs = V_ptr + n_offsets[:, None] * stride_vn + d_offsets[None, :] * stride_vd
        v = tl.load(v_ptrs, mask=n_mask[:, None] & d_mask[None, :], other=0.0)
        
        # Rescale accumulator and add new contribution
        acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)
        
        # Update max
        m_i = m_new
    
    # ---- Finalize: divide by sum of exp ----
    acc = acc / l_i[:, None]
    
    # ---- Store output ----
    o_ptrs = O_ptr + m_offsets[:, None] * stride_on + d_offsets[None, :] * stride_od
    tl.store(o_ptrs, acc, mask=m_mask[:, None] & d_mask[None, :])
    
    # Store log-sum-exp for backward pass
    l_ptrs = L_ptr + m_offsets
    lse = m_i + tl.log(l_i)  # log(sum(exp(s - m))) + m = log(sum(exp(s)))
    tl.store(l_ptrs, lse, mask=m_mask)


print("FlashAttention forward kernel defined.")
print("\nKey implementation details:")
print("  - Each program handles BLOCK_M rows of Q")
print("  - Iterates over all K/V blocks (inner loop)")
print("  - Uses online softmax for numerical stability")
print("  - Supports causal masking")
print("  - Stores log-sum-exp for backward pass")

In [ ]:
# Wrapper function
def flash_attention_triton(Q, K, V, causal=False):
    """
    Flash Attention using Triton.
    
    Args:
        Q, K, V: (N, d) tensors (single head, single batch)
        causal: whether to apply causal mask
    """
    N, d = Q.shape
    scale = 1.0 / (d ** 0.5)
    
    O = torch.empty_like(Q)
    L = torch.empty(N, device=Q.device, dtype=torch.float32)  # Log-sum-exp
    
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    
    grid = ((N + BLOCK_M - 1) // BLOCK_M,)
    
    flash_attention_fwd_kernel[grid](
        Q, K, V, O, L,
        N, d, scale,
        Q.stride(0), Q.stride(1),
        K.stride(0), K.stride(1),
        V.stride(0), V.stride(1),
        O.stride(0), O.stride(1),
        IS_CAUSAL=causal,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
    )
    return O, L


if torch.cuda.is_available():
    print("Testing FlashAttention Triton implementation")
    print("=" * 50)
    
    for N in [128, 256, 512]:
        for d in [64, 128]:
            for causal in [False, True]:
                Q = torch.randn(N, d, device='cuda', dtype=torch.float32)
                K = torch.randn(N, d, device='cuda', dtype=torch.float32)
                V = torch.randn(N, d, device='cuda', dtype=torch.float32)
                
                try:
                    O_flash, _ = flash_attention_triton(Q, K, V, causal=causal)
                    
                    # Reference
                    scale = 1.0 / (d ** 0.5)
                    S = Q @ K.T * scale
                    if causal:
                        mask = torch.triu(torch.ones(N, N, device='cuda'), diagonal=1).bool()
                        S.masked_fill_(mask, float('-inf'))
                    P = torch.softmax(S, dim=-1)
                    O_ref = P @ V
                    
                    error = (O_flash - O_ref).abs().max().item()
                    status = "PASS" if error < 0.01 else "FAIL"
                    print(f"  N={N:4d}, d={d:3d}, causal={str(causal):5s}: error={error:.2e} [{status}]")
                except Exception as e:
                    print(f"  N={N:4d}, d={d:3d}, causal={str(causal):5s}: ERROR - {e}")

In [ ]:
# Benchmark: FlashAttention vs Standard Attention
import time

def benchmark_attention(N, d, causal=False, n_iter=100):
    Q = torch.randn(N, d, device='cuda', dtype=torch.float32)
    K = torch.randn(N, d, device='cuda', dtype=torch.float32)
    V = torch.randn(N, d, device='cuda', dtype=torch.float32)
    scale = 1.0 / (d ** 0.5)
    
    # Warmup
    for _ in range(10):
        _ = flash_attention_triton(Q, K, V, causal=causal)
        S = Q @ K.T * scale
        P = torch.softmax(S, dim=-1)
        _ = P @ V
    
    # Benchmark FlashAttention
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        O_flash, _ = flash_attention_triton(Q, K, V, causal=causal)
    torch.cuda.synchronize()
    flash_time = (time.perf_counter() - start) / n_iter * 1000
    
    # Benchmark standard attention
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        S = Q @ K.T * scale
        if causal:
            mask = torch.triu(torch.ones(N, N, device='cuda'), diagonal=1).bool()
            S.masked_fill_(mask, float('-inf'))
        P = torch.softmax(S, dim=-1)
        O_std = P @ V
    torch.cuda.synchronize()
    std_time = (time.perf_counter() - start) / n_iter * 1000
    
    return flash_time, std_time


if torch.cuda.is_available():
    print(f"{'N':>6} | {'d':>4} | {'Causal':>6} | {'Standard (ms)':>13} | {'Flash (ms)':>10} | {'Speedup':>8}")
    print("-" * 65)
    
    for N in [256, 512, 1024, 2048]:
        for d in [64, 128]:
            try:
                flash_t, std_t = benchmark_attention(N, d, causal=True)
                print(f"{N:>6} | {d:>4} | {'True':>6} | {std_t:>13.3f} | {flash_t:>10.3f} | {std_t/flash_t:>7.2f}x")
            except Exception as e:
                print(f"{N:>6} | {d:>4} | {'True':>6} | Error: {e}")

## 7. Batched Multi-Head FlashAttention

The kernel above handles a single head. For production use, we need batch and multi-head support. The approach is simple: add batch and head dimensions to the grid.

In [ ]:
# Multi-head version (conceptual)
def flash_attention_multihead(Q, K, V, causal=False):
    """
    Multi-head FlashAttention.
    
    In practice, this would be a single kernel with grid = (num_q_blocks, batch * num_heads).
    Here we loop over heads for clarity.
    
    Args:
        Q: (batch, num_heads, seq_len, head_dim)
        K: (batch, num_kv_heads, seq_len, head_dim)
        V: (batch, num_kv_heads, seq_len, head_dim)
    """
    batch, num_heads, N, d = Q.shape
    num_kv_heads = K.shape[1]
    heads_per_kv = num_heads // num_kv_heads
    
    O = torch.empty_like(Q)
    
    for b in range(batch):
        for h in range(num_heads):
            kv_h = h // heads_per_kv  # GQA mapping
            O_bh, _ = flash_attention_triton(
                Q[b, h], K[b, kv_h], V[b, kv_h], causal=causal
            )
            O[b, h] = O_bh
    
    return O


if torch.cuda.is_available():
    batch, num_heads, N, d = 2, 4, 128, 64
    num_kv_heads = 2  # GQA: 2 query heads per KV head
    
    Q = torch.randn(batch, num_heads, N, d, device='cuda')
    K = torch.randn(batch, num_kv_heads, N, d, device='cuda')
    V = torch.randn(batch, num_kv_heads, N, d, device='cuda')
    
    try:
        O = flash_attention_multihead(Q, K, V, causal=True)
        print(f"Multi-head FlashAttention output shape: {O.shape}")
        print(f"GQA: {num_heads} query heads, {num_kv_heads} KV heads")
    except Exception as e:
        print(f"Error: {e}")

## 8. Step-by-Step FlashAttention Execution Trace

Let's trace through the algorithm with a tiny example to build intuition.

In [ ]:
import torch
import numpy as np

def trace_flash_attention(N=8, d=4, Br=4, Bc=4):
    """
    Trace through FlashAttention with a tiny example,
    showing every intermediate value.
    """
    torch.manual_seed(42)
    Q = torch.randn(N, d)
    K = torch.randn(N, d)
    V = torch.randn(N, d)
    scale = 1.0 / (d ** 0.5)
    
    print(f"FlashAttention Trace: N={N}, d={d}, Br={Br}, Bc={Bc}")
    print(f"Number of Q blocks: {N // Br}")
    print(f"Number of K/V blocks: {N // Bc}")
    print("=" * 70)
    
    O = torch.zeros(N, d)
    
    for i_block in range(N // Br):
        i_start = i_block * Br
        i_end = i_start + Br
        Qi = Q[i_start:i_end]  # (Br, d)
        
        mi = torch.full((Br,), float('-inf'))
        li = torch.zeros(Br)
        Oi = torch.zeros(Br, d)
        
        print(f"\n--- Q Block {i_block} (rows {i_start}-{i_end-1}) ---")
        
        for j_block in range(N // Bc):
            j_start = j_block * Bc
            j_end = j_start + Bc
            Kj = K[j_start:j_end]
            Vj = V[j_start:j_end]
            
            # Compute S = Q_i @ K_j^T
            Sij = (Qi @ Kj.T) * scale
            
            # Online softmax
            mij = Sij.max(dim=-1).values
            mi_new = torch.maximum(mi, mij)
            
            alpha = torch.exp(mi - mi_new)
            Pij = torch.exp(Sij - mi_new.unsqueeze(-1))
            
            li = li * alpha + Pij.sum(dim=-1)
            Oi = Oi * alpha.unsqueeze(-1) + Pij @ Vj
            mi = mi_new
            
            print(f"  K/V Block {j_block} (cols {j_start}-{j_end-1}):")
            print(f"    S[{i_block},{j_block}] scores: {Sij[0].tolist()[:4]}")
            print(f"    block_max: {mij.tolist()[:4]}")
            print(f"    running_max: {mi.tolist()[:4]}")
            print(f"    running_sum: {li.tolist()[:4]}")
        
        O[i_start:i_end] = Oi / li.unsqueeze(-1)
        print(f"  Output rows {i_start}-{i_end-1}: {O[i_start].tolist()[:4]}...")
    
    # Verify
    S_full = Q @ K.T * scale
    P_full = torch.softmax(S_full, dim=-1)
    O_ref = P_full @ V
    
    error = (O - O_ref).abs().max().item()
    print(f"\nMax error vs standard attention: {error:.2e}")
    print(f"CORRECT: {error < 1e-5}")

trace_flash_attention()

## 9. Summary

| Concept | Key Insight |
|---------|------------|
| **Memory bottleneck** | Standard attention materializes O(N^2) matrix, causing memory-bound performance |
| **Tiling** | Process Q and K/V in blocks that fit in SRAM |
| **Online softmax** | Incrementally compute softmax as K/V blocks stream through |
| **IO complexity** | O(N^2 d^2 / M) vs O(N^2) — significant reduction |
| **FA2 improvement** | Outer loop over Q (not K/V) eliminates atomics, increases parallelism |
| **FA3 on Hopper** | TMA, WGMMA, warp specialization for 1.5-2x over FA2 |
| **Causal masking** | Early termination of inner loop; skip K/V blocks after query position |

### Practical Impact

FlashAttention is used in virtually every production LLM system:
- **vLLM**: Uses FlashAttention for prefill, PagedAttention for decode
- **SGLang**: Same approach with RadixAttention for prefix caching
- **PyTorch**: `torch.nn.functional.scaled_dot_product_attention` uses FlashAttention
- **HuggingFace**: Transformers integrates FlashAttention-2

## Exercises

### Exercise 1: Implement FlashAttention with Variable Sequence Lengths

Modify the kernel to handle batched sequences with different lengths (used in prefill with padding).

### Exercise 2: Add Sliding Window Causal Mask

Implement sliding window attention within FlashAttention: each query only attends to the last W keys.

### Exercise 3: FlashAttention with Softcapping

Some models (Gemma 2) use softcapping: `scores = tanh(scores / cap) * cap` before softmax. Implement this.

### Exercise 4: Measure SRAM Usage

For different BLOCK_M and BLOCK_N values, calculate the SRAM usage and find the optimal configuration for your GPU.

In [ ]:
# Exercise helper: SRAM usage calculator
def calculate_sram_usage(BLOCK_M, BLOCK_N, d, dtype_bytes=4):
    """
    Calculate SRAM (shared memory) usage for FlashAttention.
    """
    q_tile = BLOCK_M * d * dtype_bytes        # Q block
    k_tile = BLOCK_N * d * dtype_bytes        # K block
    v_tile = BLOCK_N * d * dtype_bytes        # V block
    s_tile = BLOCK_M * BLOCK_N * dtype_bytes  # Attention scores
    state = BLOCK_M * (1 + 1 + d) * dtype_bytes  # m, l, acc
    
    total = q_tile + k_tile + v_tile + s_tile + state
    
    print(f"BLOCK_M={BLOCK_M}, BLOCK_N={BLOCK_N}, d={d}:")
    print(f"  Q tile:    {q_tile/1024:6.1f} KB")
    print(f"  K tile:    {k_tile/1024:6.1f} KB")
    print(f"  V tile:    {v_tile/1024:6.1f} KB")
    print(f"  S tile:    {s_tile/1024:6.1f} KB")
    print(f"  State:     {state/1024:6.1f} KB")
    print(f"  TOTAL:     {total/1024:6.1f} KB")
    print(f"  Fits in 192KB SRAM: {'YES' if total < 192*1024 else 'NO'}")
    print()

for bm, bn in [(32, 32), (64, 64), (128, 64), (128, 128)]:
    calculate_sram_usage(bm, bn, d=128)

In [ ]:
# Exercise 2 Skeleton: Sliding Window
print("Exercise 2: Sliding Window FlashAttention")
print()
print("Modifications to the kernel:")
print("1. Add WINDOW_SIZE parameter")
print("2. Change inner loop start:")
print("   kv_start = max(0, q_block_start - WINDOW_SIZE)")
print("   # Round down to block boundary:")
print("   kv_start = (kv_start // BLOCK_N) * BLOCK_N")
print("3. Apply window mask within the first/last blocks:")
print("   window_mask = (m_offsets[:, None] - n_offsets[None, :]) < WINDOW_SIZE")
print("   s = tl.where(causal_mask & window_mask, s, -inf)")